### Spiegel dataset fullness check 

In [ ]:
import pandas as pd

df = pd.read_csv("spiegel_corpus.csv")

print("="*80)
print("ROWS")
print(len(df))

print("\nCOLUMNS")
for c in df.columns:
    print(c)

print("\n" + "="*80)

# -----------------------------
# MISSING VALUES
# -----------------------------
summary = []

for col in df.columns:

    missing = df[col].isna().sum()

    empty = 0

    if df[col].dtype == object:
        empty = (
            df[col]
            .fillna("")
            .str.strip()
            .eq("")
            .sum()
        )

    summary.append({
        "column": col,
        "missing": missing,
        "empty_strings": empty,
        "filled": len(df) - max(missing, empty)
    })

summary = pd.DataFrame(summary)

print(summary.sort_values("missing", ascending=False))

print("\n" + "="*80)

# -----------------------------
# SHOW IMPORTANT FIELDS
# -----------------------------
important = [
    "article_url",
    "publication_date",
    "title",
    "headline",
    "caption",
    "article_text",
    "image_urls",
    "image_count",
    "section"
]

print("\nIMPORTANT FIELDS\n")

for field in important:

    if field in df.columns:

        print(f"{field:20}  OK")

    else:

        print(f"{field:20}  MISSING")

In [ ]:
df.head()

In [ ]:
print(df["article_text"].iloc[0][:1000])

df["article_text"].str.contains(r"russland", case=False, na=False).sum()

df["article_text"].str.contains(r"putin", case=False, na=False).sum()

In [ ]:
# filter articles about russia

russia_keywords = [
    "russland",
    "russisch",
    "russe",
    "russen",
    "kreml",
    "kremlin",
    "moskau",
    "putin",
    "lawrow",
    "peskow",
    "schoigu",
    "gerassimow",
    "medwedew",
    "sacharowa",
    "mischustin",
    "gazprom",
    "rosneft",
    "rosatom",
    "transneft",
    "lukoil",
    "duma",
    "staatsduma",
    "krim",
    "ukrainekrieg", 
    "angriffskrieg"
]

pattern = "|".join(russia_keywords)

russia_df = df[
    df["article_text"]
    .fillna("")
    .str.contains(pattern, case=False, na=False)
].copy()

print(len(russia_df))

In [ ]:
# count articles matched by each keyword

keyword_counts = {}

text = df["article_text"].fillna("")

for keyword in russia_keywords:
    keyword_counts[keyword] = text.str.contains(
        keyword,
        case=False,
        na=False
    ).sum()

keyword_counts = (
    pd.Series(keyword_counts)
    .sort_values(ascending=False)
    .reset_index()
)

keyword_counts.columns = ["keyword", "articles"]

display(keyword_counts)

In [ ]:
# compare filtered corpus with old high recall corpus

old_df = pd.read_csv("spiegel_ukraine_russia_high_recall.csv")

old_urls = set(old_df["article_url"])
new_urls = set(russia_df["article_url"])

print(f"old corpus: {len(old_urls)}")
print(f"new corpus: {len(new_urls)}")

In [ ]:
# compare overlap

print(f"overlap: {len(old_urls & new_urls)}")
print(f"only old: {len(old_urls - new_urls)}")
print(f"only new: {len(new_urls - old_urls)}")

In [ ]:
# save final corpus after the second keyword filtering

russia_df.to_csv(
    "spiegel_SECONDROUND_final_articles.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nfinal corpus saved")

print(len(russia_df))

In [ ]:
# find new articles

missing_articles = russia_df[
    ~russia_df["article_url"].isin(old_urls)
].copy()

print(len(missing_articles))

display(
    missing_articles[
        ["published_time", "title", "article_url"]
    ].head()
)

In [ ]:
# save new articles

missing_articles.to_csv(
    "spiegel_missing_articles.csv",
    index=False
)

In [ ]:
# find articles removed by the new filter

removed_articles = old_df[
    ~old_df["article_url"].isin(new_urls)
].copy()

print(len(removed_articles))

# display full urls

pd.set_option("display.max_colwidth", None)

display(
    removed_articles[
        ["published_time", "title", "article_url"]
    ].head(20)
)

In [ ]:
# load image dataset

images_df = pd.read_csv("spiegel_clean_article_images_filtered.csv")
images_df

In [ ]:
images_df.columns

In [ ]:
# check available metadata in final spiegel corpus
df = pd.read_csv("spiegel_SECONDROUND_final_articles.csv")

print("\nrows")

print(len(df))

print("\ncolumns")

for column in df.columns:
    print(column)

print("\nmissing values")

summary = []

for column in df.columns:

    summary.append({

        "column": column,
        "missing": df[column].isna().sum(),
        "filled": df[column].notna().sum()

    })

summary = pd.DataFrame(summary)

print(summary)

In [ ]:
print(
    df[
        [
            "article_url",
            "image_count",
            "image_urls"
        ]
    ].head()
)

## New parcer for spiegel for new dataset 

In [ ]:
# load final spiegel dataset

import pandas as pd
import aiohttp
import asyncio
import json
import re
import random

from bs4 import BeautifulSoup
from tqdm.asyncio import tqdm

input_file = "spiegel_SECONDROUND_final_articles.csv"
output_file = "spiegel_final_metadata.csv"

concurrent_requests = 30
timeout = 30
save_every = 100

headers = {
    "User-Agent": (
        "Mozilla/5.0 "
        "(Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 "
        "Safari/537.36"
    )
}

df = pd.read_csv(input_file)

print(len(df))
display(df.head())

In [ ]:
# initialize metadata columns

metadata_columns = [
    "article_id",
    "section",
    "published_time",
    "caption",
    "image_count",
    "image_urls"
]

for column in metadata_columns:

    if column not in df.columns:

        df[column] = None

print(df.columns.tolist())

### test

In [ ]:
# prepare dataframe dtypes

text_columns = [

    "article_id",
    "section",
    "published_time",
    "title",
    "article_text",
    "image_urls",
    "caption"

]

for column in text_columns:

    if column in df.columns:

        df[column] = df[column].astype("object")

In [ ]:
# parse article metadata

def extract_ld_json(soup):

    scripts = soup.find_all(
        "script",
        type="application/ld+json"
    )

    for script in scripts:

        try:

            data = json.loads(script.string)

            if isinstance(data, list):

                for item in data:

                    if (
                        isinstance(item, dict)
                        and item.get("@type") == "NewsArticle"
                    ):
                        return item

            elif (
                isinstance(data, dict)
                and data.get("@type") == "NewsArticle"
            ):

                return data

        except:

            continue

    return {}


# extract metadata from parsed html

def parse_article_from_html(soup, ld, url):

    article_id = None

    match = re.search(
        r"-a-([a-zA-Z0-9\-]+)$",
        url.rstrip("/")
    )

    if match:

        article_id = match.group(1)


    section = None

    try:

        section = (
            url
            .replace("https://www.spiegel.de/", "")
            .split("/")[0]
        )

    except:

        pass


    title = None

    if ld.get("headline"):

        title = ld["headline"]

    else:

        meta = soup.find(
            "meta",
            property="og:title"
        )

        if meta:

            title = meta.get("content")

        elif soup.title:

            title = soup.title.get_text(strip=True)


    published_time = None

    if ld.get("datePublished"):

        published_time = ld["datePublished"]

    else:

        meta = soup.find(
            "meta",
            attrs={
                "property": "article:published_time"
            }
        )

        if meta:

            published_time = meta.get("content")


    paragraphs = soup.find_all("p")

    text = []

    for p in paragraphs:

        paragraph = p.get_text(
            " ",
            strip=True
        )

        if len(paragraph) > 30:

            text.append(paragraph)

    article_text = "\n".join(text)


    ignore_patterns = [

        "games-icons",
        "/public/",
        "feedback",
        "user-capping",
        "magazin.spiegel.de",
        ".svg",
        "logo",
        "icon",
        "avatar",
        "placeholder",
        "favicon"

    ]

    image_urls = []
    unique_images = {}

    for img in soup.find_all("img"):

        candidates = []

        for attribute in [

            "src",
            "data-src"

        ]:

            value = img.get(attribute)

            if value:

                candidates.append(value)

        for attribute in [

            "srcset",
            "data-srcset"

        ]:

            value = img.get(attribute)

            if value:

                for item in value.split(","):

                    candidates.append(
                        item.strip().split(" ")[0]
                    )

        for image_url in candidates:

            if image_url.startswith("//"):

                image_url = "https:" + image_url

            if not image_url.startswith("http"):

                continue

            image_url = image_url.split("?")[0]

            if any(
                pattern in image_url.lower()
                for pattern in ignore_patterns
            ):

                continue

            if not re.search(
                r"\.(jpg|jpeg|png|webp|avif)$",
                image_url.lower()
            ):

                continue

            match = re.search(
                r"/images/([a-f0-9\-]{36})",
                image_url
            )

            if match:

                image_id = match.group(1)

            else:

                image_id = image_url

            if image_id not in unique_images:

                unique_images[image_id] = image_url

    image_urls = list(unique_images.values())

    image_count = len(image_urls)


    caption = None

    figcaption = soup.find("figcaption")

    if figcaption:

        caption = figcaption.get_text(
            " ",
            strip=True
        )

        if caption == "":

            caption = None

    return {

        "article_id": article_id,
        "section": section,
        "published_time": published_time,
        "title": title,
        "article_text": article_text,
        "image_urls": json.dumps(
            image_urls,
            ensure_ascii=False
        ),
        "image_count": image_count,
        "caption": caption

    }
async def parse_article(session, url):

    try:

        async with session.get(
            url,
            headers=headers,
            timeout=timeout
        ) as response:

            html = await response.text()

    except Exception as e:

        print(url)
        print(e)

        return None


    soup = BeautifulSoup(
        html,
        "html.parser"
    )

    ld = extract_ld_json(soup)


    article_id = None

    match = re.search(
        r"-a-([a-zA-Z0-9\-]+)$",
        url.rstrip("/")
    )

    if match:

        article_id = match.group(1)


    section = None

    try:

        section = (
            url
            .replace("https://www.spiegel.de/", "")
            .split("/")[0]
        )

    except:

        pass


    title = None

    if ld.get("headline"):

        title = ld["headline"]

    else:

        meta = soup.find(
            "meta",
            property="og:title"
        )

        if meta:

            title = meta.get("content")

        elif soup.title:

            title = soup.title.get_text(strip=True)


    published_time = None

    if ld.get("datePublished"):

        published_time = ld["datePublished"]

    else:

        meta = soup.find(
            "meta",
            attrs={
                "property": "article:published_time"
            }
        )

        if meta:

            published_time = meta.get("content")


    paragraphs = soup.find_all("p")

    text = []

    for p in paragraphs:

        paragraph = p.get_text(
            " ",
            strip=True
        )

        if len(paragraph) > 30:

            text.append(paragraph)

    article_text = "\n".join(text)


    ignore_patterns = [

        "games-icons",
        "/public/",
        "feedback",
        "user-capping",
        "magazin.spiegel.de",
        ".svg",
        "logo",
        "icon",
        "avatar",
        "placeholder",
        "favicon"

    ]


    image_urls = []
    unique_images = {}

    for img in soup.find_all("img"):

        candidates = []

        for attribute in [

            "src",
            "data-src"

        ]:

            value = img.get(attribute)

            if value:

                candidates.append(value)

        for attribute in [

            "srcset",
            "data-srcset"

        ]:

            value = img.get(attribute)

            if value:

                for item in value.split(","):

                    candidates.append(
                        item.strip().split(" ")[0]
                    )


        for image_url in candidates:

            if image_url.startswith("//"):

                image_url = "https:" + image_url

            if not image_url.startswith("http"):

                continue

            image_url = image_url.split("?")[0]

            if any(
                pattern in image_url.lower()
                for pattern in ignore_patterns
            ):

                continue

            if not re.search(
                r"\.(jpg|jpeg|png|webp|avif)$",
                image_url.lower()
            ):

                continue

            match = re.search(

                r"/images/([a-f0-9\-]{36})",

                image_url

            )

            if match:

                image_id = match.group(1)

            else:

                image_id = image_url


            if image_id not in unique_images:

                unique_images[image_id] = image_url


    image_urls = list(unique_images.values())

    image_count = len(image_urls)


    caption = None

    figcaption = soup.find("figcaption")

    if figcaption:

        caption = figcaption.get_text(
            " ",
            strip=True
        )

        if caption == "":

            caption = None


    return {

        "article_id": article_id,

        "section": section,

        "published_time": published_time,

        "title": title,

        "article_text": article_text,

        "image_urls": json.dumps(
            image_urls,
            ensure_ascii=False
        ),

        "image_count": image_count,

        "caption": caption

    }

In [ ]:
# test parser

async def test_parser():

    async with aiohttp.ClientSession(
        headers=headers
    ) as session:

        result = await parse_article(
            session,
            df.loc[0, "article_url"]
        )

        return result


result = await test_parser()

result

In [ ]:
# download metadata for all articles

import requests
from tqdm import tqdm

session = requests.Session()
session.headers.update(headers)

processed = 0

for index in tqdm(range(len(df))):

    if (
        pd.notna(df.loc[index, "published_time"])
        and pd.notna(df.loc[index, "article_id"])
    ):
        continue

    url = df.loc[index, "article_url"]

    try:

        response = session.get(
            url,
            timeout=30
        )

        response.raise_for_status()

        html = response.text

        soup = BeautifulSoup(
            html,
            "html.parser"
        )

        ld = extract_ld_json(soup)

        result = parse_article_from_html(
            soup,
            ld,
            url
        )

        for column, value in result.items():

            df.loc[index, column] = value

    except Exception as e:

        print(url)
        print(e)

    processed += 1

    if processed % save_every == 0:

        df.to_csv(
            output_file,
            index=False,
            encoding="utf-8-sig"
        )

df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

session.close()

print("finished")

In [ ]:

import requests

from bs4 import BeautifulSoup

url = df.loc[0, "article_url"]

print(url)

response = requests.get(
    url,
    headers=headers,
    timeout=30
)

print("status:", response.status_code)

html = response.text

print("html length:", len(html))

soup = BeautifulSoup(
    html,
    "html.parser"
)

print()

print("json-ld:", len(soup.select('script[type="application/ld+json"]')))

print("meta og:image:", len(soup.select('meta[property="og:image"]')))

print("picture:", len(soup.select("picture")))

print("source:", len(soup.select("picture source")))

print("img:", len(soup.find_all("img")))

In [ ]:
import json

script = soup.select_one(
    'script[type="application/ld+json"]'
)

print(script is not None)

data = json.loads(script.string)

print(type(data))

if isinstance(data, dict):

    print(data.keys())

    print()

    print("image:")

    print(data.get("image"))

In [ ]:
import json
from pprint import pprint

script = soup.select_one(
    'script[type="application/ld+json"]'
)

data = json.loads(script.string)

print("objects:", len(data))

print()

for i, obj in enumerate(data):

    print("=" * 60)

    print(i)

    if isinstance(obj, dict):

        print(obj.get("@type"))

        print(obj.keys())

In [ ]:
import json
from pprint import pprint

script = soup.select_one(
    'script[type="application/ld+json"]'
)

data = json.loads(script.string)

article = data[0]

print(type(article["image"]))

print()

pprint(article["image"])

In [ ]:
# show all figures in the article

figures = soup.find_all("figure")

print("figures:", len(figures))

print()

for i, figure in enumerate(figures):

    print("=" * 80)

    print("figure", i)

    imgs = figure.find_all("img")

    print("img tags:", len(imgs))

    for img in imgs:

        print("src     :", img.get("src"))

        print("srcset  :", img.get("srcset"))

        print()

In [ ]:
article = soup.find("article")

print(article is not None)

if article:

    print("img:", len(article.find_all("img")))

    print("picture:", len(article.find_all("picture")))

    print("figure:", len(article.find_all("figure")))

In [ ]:
for img in article.find_all("img")[:30]:

    print("=" * 60)

    print(img.get("src"))

    print()

    print(img.get("srcset"))

    print()

    print(img.get("alt"))

In [ ]:
import re

urls = []

for img in article.find_all("img"):

    src = img.get("src")

    if isinstance(src, str) and "cdn.prod.www.spiegel.de/images/" in src:
        urls.append(src)

print("found:", len(urls))

print()

for u in urls[:20]:

    m = re.search(
        r"/images/([a-f0-9-]+)_",
        u
    )

    print("url :", u)

    if m:
        print("id  :", m.group(1))

    print()

In [ ]:
# DOWNLOAD

In [ ]:
import re
import requests
import pandas as pd

from bs4 import BeautifulSoup
from tqdm import tqdm

# =====================================================
# extract article images
# =====================================================

def extract_article_images(soup):

    article = soup.find("article")

    if article is None:
        return []

    best_images = {}

    for img in article.find_all("img"):

        src = img.get("src")

        if not isinstance(src, str):
            continue

        if not src.startswith(
            "https://cdn.prod.www.spiegel.de/images/"
        ):
            continue

        m = re.search(
            r"/images/([a-f0-9-]+)_w(\d+)",
            src
        )

        if m is None:
            continue

        image_id = m.group(1)

        width = int(m.group(2))

        if (
            image_id not in best_images
            or
            width > best_images[image_id][0]
        ):

            best_images[image_id] = (
                width,
                src
            )

    images = [

        value[1]

        for value in best_images.values()

    ]

    return images


# =====================================================
# parse all articles
# =====================================================

session = requests.Session()

session.headers.update(headers)

# overwrite old image information

df["image_urls"] = ""

df["image_count"] = 0

processed = 0

errors = []

for index in tqdm(range(len(df))):

    url = df.loc[
        index,
        "article_url"
    ]

    try:

        response = session.get(
            url,
            timeout=30
        )

        response.raise_for_status()

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        images = extract_article_images(
            soup
        )

        df.loc[
            index,
            "image_urls"
        ] = "|".join(images)

        df.loc[
            index,
            "image_count"
        ] = len(images)

    except Exception as e:

        errors.append({

            "row": index,

            "article_url": url,

            "error": str(e)

        })

        df.loc[
            index,
            "image_urls"
        ] = ""

        df.loc[
            index,
            "image_count"
        ] = 0

    processed += 1

    if processed % save_every == 0:

        df.to_csv(

            output_file,

            index=False,

            encoding="utf-8-sig"

        )

# =====================================================
# save final corpus
# =====================================================

df.to_csv(

    output_file,

    index=False,

    encoding="utf-8-sig"

)

# =====================================================
# save errors
# =====================================================

errors_df = pd.DataFrame(errors)

errors_df.to_csv(

    "spiegel_image_errors.csv",

    index=False,

    encoding="utf-8-sig"

)

session.close()

# =====================================================
# diagnostics
# =====================================================

total_links = 0

unique_links = set()

for urls in df["image_urls"]:

    if not urls:
        continue

    split_urls = [

        u.strip()

        for u in str(urls).split("|")

        if u.strip()

    ]

    total_links += len(split_urls)

    unique_links.update(split_urls)

print()

print("=" * 60)

print(f"articles processed : {processed:,}")

print(f"errors             : {len(errors):,}")

print(f"total image urls   : {total_links:,}")

print(f"unique image urls  : {len(unique_links):,}")

print(f"average/article    : {total_links / len(df):.2f}")

print()

print("saved corpus:")

print(output_file)

print()

print("saved errors:")

print("spiegel_image_errors.csv")

In [ ]:
import pandas as pd
from collections import Counter

# =====================================================
# load metadata
# =====================================================

df = pd.read_csv(
    "spiegel_final_metadata.csv",
    low_memory=False
)

# =====================================================
# collect image urls
# =====================================================

all_urls = []

for urls in df["image_urls"].fillna(""):

    if urls == "":
        continue

    for url in str(urls).split("|"):

        url = url.strip()

        if url:

            all_urls.append(url)

# =====================================================
# count occurrences
# =====================================================

counts = Counter(all_urls)

# =====================================================
# keep images appearing <= 5 times
# =====================================================

filtered_urls = [

    url

    for url, n

    in counts.items()

    if n <= 5

]

# =====================================================
# save
# =====================================================

unique_images = pd.DataFrame({

    "image_url": filtered_urls,

    "occurrences": [

        counts[url]

        for url

        in filtered_urls

    ]

})

unique_images.to_csv(

    "spiegel_unique_imagesSECONDROUND.csv",

    index=False,

    encoding="utf-8-sig"

)

# =====================================================
# diagnostics
# =====================================================

print()

print("=" * 60)

print("all image urls:")

print(f"{len(all_urls):,}")

print()

print("unique image urls:")

print(f"{len(counts):,}")

print()

print("kept (<=5 occurrences):")

print(f"{len(unique_images):,}")

print()

print("removed (>5 occurrences):")

print(f"{len(counts)-len(unique_images):,}")

print()

print("saved:")

print("spiegel_unique_imagesSECONDROUND.csv")

In [ ]:
import re
import pandas as pd

# =====================================================
# FILES
# =====================================================

METADATA_FILE = "spiegel_final_metadata.csv"

UNIQUE_IMAGES_FILE = "spiegel_unique_imagesSECONDROUND.csv"

# =====================================================
# LOAD
# =====================================================

metadata = pd.read_csv(
    METADATA_FILE,
    low_memory=False
)

unique_images = pd.read_csv(
    UNIQUE_IMAGES_FILE
)

# =====================================================
# URL -> FILENAME
# =====================================================

def url_to_filename(url):

    if pd.isna(url):
        return ""

    url = str(url)

    m = re.search(

        r"/images/([a-f0-9-]+)_",

        url

    )

    if m is None:

        return ""

    image_id = m.group(1)

    if ".png" in url.lower():

        ext = ".png"

    elif ".webp" in url.lower():

        ext = ".webp"

    else:

        ext = ".jpg"

    return image_id + ext

# =====================================================
# UNIQUE IMAGES
# =====================================================

unique_images["filename"] = (

    unique_images["image_url"]

    .apply(url_to_filename)

)

unique_images.to_csv(

    UNIQUE_IMAGES_FILE,

    index=False,

    encoding="utf-8-sig"

)

# =====================================================
# METADATA
# =====================================================

def urls_to_filenames(urls):

    if pd.isna(urls):

        return ""

    filenames = []

    for url in str(urls).split("|"):

        name = url_to_filename(url)

        if name:

            filenames.append(name)

    return "|".join(filenames)

metadata["image_filenames"] = (

    metadata["image_urls"]

    .apply(urls_to_filenames)

)

metadata.to_csv(

    METADATA_FILE,

    index=False,

    encoding="utf-8-sig"

)

# =====================================================
# DIAGNOSTICS
# =====================================================

print()

print("=" * 60)

print("metadata rows:")

print(f"{len(metadata):,}")

print()

print("unique images:")

print(f"{len(unique_images):,}")

print()

print("filenames created:")

print(unique_images["filename"].ne("").sum())

print()

print("saved:")

print(METADATA_FILE)

print(UNIQUE_IMAGES_FILE)

In [ ]:
import os
import re
import time
import requests
import pandas as pd

from tqdm import tqdm
from concurrent.futures import (
    ThreadPoolExecutor,
    as_completed
)

# =====================================================
# SETTINGS
# =====================================================

INPUT_FILE = (
    "spiegel_unique_imagesSECONDROUND.csv"
)

OUTPUT_DIR = (
    "spiegel_images_SECONDROUND"
)

FAILED_FILE = (
    "spiegel_failed_imagesSECONDROUND.csv"
)

MAX_WORKERS = 10

MAX_RETRIES = 3

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

HEADERS = {

    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )

}

# =====================================================
# EXTRACT IMAGE UUID
# =====================================================

def extract_image_id(url):

    m = re.search(

        r"/images/([a-f0-9-]+)_",

        str(url)

    )

    if m:

        return m.group(1)

    return None

# =====================================================
# EXTENSION
# =====================================================

def get_extension(url):

    url = str(url).lower()

    if ".png" in url:
        return ".png"

    if ".webp" in url:
        return ".webp"

    return ".jpg"

# =====================================================
# DOWNLOAD
# =====================================================

def download_image(row):

    url = row["image_url"]

    image_id = extract_image_id(url)

    if image_id is None:

        return (
            False,
            url
        )

    ext = get_extension(url)

    filename = image_id + ext

    filepath = os.path.join(
        OUTPUT_DIR,
        filename
    )

    if os.path.exists(filepath):

        return (
            True,
            "already_exists"
        )

    for _ in range(MAX_RETRIES):

        try:

            response = requests.get(
                url,
                headers=HEADERS,
                timeout=60
            )

            if response.status_code == 200:

                with open(
                    filepath,
                    "wb"
                ) as f:

                    f.write(
                        response.content
                    )

                return (
                    True,
                    None
                )

        except Exception:
            pass

        time.sleep(2)

    return (
        False,
        url
    )

# =====================================================
# LOAD
# =====================================================

df = pd.read_csv(
    INPUT_FILE
)

print()

print("images to download:")

print(f"{len(df):,}")

# =====================================================
# DOWNLOAD
# =====================================================

downloaded = 0

already_exists = 0

failed = []

with ThreadPoolExecutor(
    max_workers=MAX_WORKERS
) as executor:

    futures = {

        executor.submit(
            download_image,
            row
        ): row

        for _, row
        in df.iterrows()

    }

    for future in tqdm(

        as_completed(futures),

        total=len(futures)

    ):

        success, info = future.result()

        if success:

            if info == "already_exists":

                already_exists += 1

            else:

                downloaded += 1

        else:

            failed.append({

                "image_url": info,

                "filename":
                    os.path.basename(info)

            })

# =====================================================
# SAVE FAILURES
# =====================================================

pd.DataFrame(
    failed
).to_csv(

    FAILED_FILE,

    index=False,

    encoding="utf-8-sig"

)

# =====================================================
# SUMMARY
# =====================================================

print()

print("=" * 60)

print(f"downloaded: {downloaded:,}")

print(f"already existed: {already_exists:,}")

print(f"failed: {len(failed):,}")

print()

print("saved folder:")

print(OUTPUT_DIR)

print()

print("failed file:")

print(FAILED_FILE)

In [ ]:
import os

print(len(os.listdir("spiegel_images_SECONDROUND")))

In [ ]:
import pandas as pd

df = pd.read_csv("spiegel_unique_imagesSECONDROUND.csv")

print(df.head())

print(df.columns)

print()

print("rows:", len(df))

print()

print("unique image_url:", df["image_url"].nunique())

print("unique filename :", df["filename"].nunique())

In [ ]:
import os
import pandas as pd

csv = pd.read_csv(
    "spiegel_unique_imagesSECONDROUND.csv"
)

files = [

    f

    for f in os.listdir(
        "spiegel_images_SECONDROUND"
    )

    if f.lower().endswith((
        ".jpg",
        ".jpeg",
        ".png",
        ".webp"
    ))
]

print()

print("rows in csv:")

print(f"{len(csv):,}")

print()

print("unique filenames in csv:")

print(csv["filename"].nunique())

print()

print("files in folder:")

print(f"{len(files):,}")

print()

print("unique files in folder:")

print(len(set(files)))

In [ ]:
import re
import pandas as pd

df = pd.read_csv(
    "spiegel_unique_imagesSECONDROUND.csv"
)

# ---------------------------------------

def width(url):

    m = re.search(
        r"_w(\d+)",
        str(url)
    )

    if m:

        return int(
            m.group(1)
        )

    return 0

# ---------------------------------------

df["width"] = (

    df["image_url"]

    .apply(width)

)

df = (

    df

    .sort_values(

        "width",

        ascending=False

    )

    .drop_duplicates(

        subset="filename",

        keep="first"

    )

    .sort_values(

        "filename"

    )

    .drop(

        columns="width"

    )

)

df.to_csv(

    "spiegel_unique_imagesSECONDROUND.csv",

    index=False,

    encoding="utf-8-sig"

)

print()

print("unique images:")

print(f"{len(df):,}")

print()

print("saved:")

print("spiegel_unique_imagesSECONDROUND.csv")

In [ ]:
import re
import pandas as pd

# add image filenames to the final metadata
#
# converts every image URL into its corresponding local filename.
# one article may contain multiple images, stored as "|" separated values.
# the same image can appear in different articles, which is preserved.
#
# output:
# spiegel_final_metadataSECONDROUND.csv

INPUT_FILE = "spiegel_final_metadata.csv"

OUTPUT_FILE = "spiegel_final_metadataSECONDROUND.csv"

# load metadata

df = pd.read_csv(
    INPUT_FILE,
    low_memory=False
)

# convert image url to filename

def url_to_filename(url):

    if pd.isna(url):

        return ""

    url = str(url).strip()

    match = re.search(
        r"/images/([a-f0-9-]+)_",
        url
    )

    if match is None:

        return ""

    image_id = match.group(1)

    if ".png" in url.lower():

        extension = ".png"

    elif ".webp" in url.lower():

        extension = ".webp"

    else:

        extension = ".jpg"

    return image_id + extension

# convert all image urls of one article

def urls_to_filenames(urls):

    if pd.isna(urls):

        return ""

    filenames = []

    seen = set()

    for url in str(urls).split("|"):

        filename = url_to_filename(url)

        if filename == "":

            continue

        if filename in seen:

            continue

        seen.add(filename)

        filenames.append(filename)

    return "|".join(filenames)

# create image_filenames column

df["image_filenames"] = (

    df["image_urls"]

    .apply(urls_to_filenames)

)

# save updated metadata

df.to_csv(

    OUTPUT_FILE,

    index=False,

    encoding="utf-8-sig"

)

# diagnostics

total_images = 0

for value in df["image_filenames"]:

    if pd.isna(value):

        continue

    if value.strip() == "":

        continue

    total_images += len(
        value.split("|")
    )

print()

print("articles:")

print(f"{len(df):,}")

print()

print("image occurrences:")

print(f"{total_images:,}")

print()

print("average images per article:")

print(f"{total_images / len(df):.2f}")

print()

print("saved:")

print(OUTPUT_FILE)